In [1]:
# Importing Modules
import torch
import pandas as pd
import numpy as np
import itertools
from src.dataloader import loadDataGNN_FP
from src.utils.smiles2morganfp import MorganFP
from src.model import SimpleGCN_FP, SimpleGAT_FP, SimpleGIN_FP, SimpleGraphSAGE_FP
from src.train_test import TrainGNN_FP, TestGNN_FP
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"])
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"])
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

esol_train_data = esol_train_fp
esol_test_data = esol_test_fp

######################## DATA-2 ##################################
# Loading FreeSolv data
freeSolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freeSolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

# Generate FreeSolv FP
freeSolv_train_fp = MorganFP(freeSolv_train_data["smiles"])
freeSolv_train_fp["smiles"] = freeSolv_train_fp.index
freeSolv_train_fp = freeSolv_train_fp.merge(freeSolv_train_data, on="smiles")
freeSolv_test_fp = MorganFP(freeSolv_test_data["smiles"])
freeSolv_test_fp["smiles"] = freeSolv_test_fp.index
freeSolv_test_fp = freeSolv_test_fp.merge(freeSolv_test_data, on="smiles")


freeSolv_train_data = freeSolv_train_fp
freeSolv_test_data = freeSolv_test_fp

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"])
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"])
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

lipophilicity_train_data = lipophilicity_train_fp
lipophilicity_test_data = lipophilicity_test_fp

In [3]:
# Models
model_dict = {
        "SimpleGCN_FP":SimpleGCN_FP, 
		"SimpleGAT_FP":SimpleGAT_FP, 
		"SimpleGIN_FP":SimpleGIN_FP,
		"SimpleGraphSAGE_FP":SimpleGraphSAGE_FP
}

# Models Hyperparameters
model_hyperparams_dict = {
        "SimpleGCN_FP":[0.01, 64, 8],
        "SimpleGAT_FP":[0.005, 128, 32],
        "SimpleGIN_FP":[0.001, 64, 4],
        "SimpleGraphSAGE_FP":[0.005, 128, 4]
}

# Function to run analysis
def RunGNN(model, train_data, test_data, dataName, modelName, params):

    # Params
	lr, h_dim, b_size = params

	# Loading dataset
	train_loader = loadDataGNN_FP(train_data, batch_size=b_size)
	test_loader = loadDataGNN_FP(test_data, batch_size=b_size)

	# Model
	model = model_dict[modelName](num_features=8, hidden_dim=h_dim, num_classes=1)

	# Model training
	model = TrainGNN_FP(model, train_loader, epochs=100, learning_rate=lr)

	# Model testing
	y_test, y_pred = TestGNN_FP(model, test_loader)

	# Calculate MAE
	mae = mean_absolute_error(y_test, y_pred)

	# Calculate RMSE
	rmse = root_mean_squared_error(y_test, y_pred)
    
    # Calculate pearson r and p_val
	r, p_val = pearsonr(y_test, y_pred)
    
	# Return results
	return pd.DataFrame({"Data":[dataName], "Model":[modelName],
			"MAE":round(mae, 2), "RMSE":[round(rmse, 2)], 
			"r":[round(r, 2)], "p-val":[round(p_val, 3)]})

In [4]:
# Data dict
datasets = {
    "ESOL":{"train":esol_train_data, "test":esol_test_data},
    "FreeSolv":{"train":freeSolv_train_data, "test":freeSolv_test_data},
    "Lipophilicity":{"train":lipophilicity_train_data, "test":lipophilicity_test_data}
}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        temp_out.append(RunGNN(model, data["train"], data["test"], dataName, modelName, model_hyperparams_dict[modelName]))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_FP_Analysis.csv")
GNN_out

,Data,Model,MAE,RMSE,r,p-val
0,ESOL,SimpleGCN_FP,0.85,1.14,0.86,0.0
1,FreeSolv,SimpleGCN_FP,1.21,2.00,0.88,0.0
2,Lipophilicity,SimpleGCN_FP,0.67,0.90,0.71,0.0
3,ESOL,SimpleGAT_FP,0.81,1.07,0.87,0.0
4,FreeSolv,SimpleGAT_FP,1.18,2.01,0.88,0.0
5,Lipophilicity,SimpleGAT_FP,0.55,0.75,0.79,0.0
6,ESOL,SimpleGIN_FP,0.78,1.05,0.88,0.0
7,FreeSolv,SimpleGIN_FP,1.17,2.03,0.87,0.0
8,Lipophilicity,SimpleGIN_FP,0.54,0.74,0.80,0.0
9,ESOL,SimpleGraphSAGE_FP,0.83,1.11,0.86,0.0


In [6]:
# Function to plot barplot showing result
def plot_bar(data, target):
    sns.set_theme(style="ticks", context="paper")
    g = sns.catplot(data=data, kind="bar", x="Model", y=target, hue="Model",
                    col="Data", col_wrap=3, sharey=False, height=4, width=0.8,
                    aspect=0.8, dodge=False)
    g.set_axis_labels("Model", f"{target}")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.tick_params(axis="x", rotation=40)
    plt.tight_layout()
    plt.savefig(f"results/GNN_FP_Analysis_{target}_Plot.png", dpi=300)
    plt.close()

In [7]:
# Plotting barplot for RMSE
plot_bar(GNN_out, "RMSE")
# Plotting barplot for MAE
plot_bar(GNN_out, "MAE")
# Plotting barplot for 
plot_bar(GNN_out, "r")